# Bài 4: Mô hình phân lớp Iris bằng Neural Network (Keras) với k-fold cross validation

Notebook này thực hiện **câu c** và **câu d**. Để chạy được hai câu này, bên dưới có một ô chuẩn bị dữ liệu ngắn (đọc bộ dữ liệu Iris và chuẩn hóa `data` về đoạn [0,1], tương ứng câu a, b).

## Chuẩn bị dữ liệu (cần cho câu c, d)

In [1]:
print("Họ và tên: Lê Quốc Trung - MSSV: 6451071082")
import numpy as np
from sklearn.datasets import load_iris
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow import keras
from tensorflow.keras import layers

# Đọc bộ dữ liệu Iris
iris = load_iris()
X = iris.data
y = iris.target

# Chuẩn hóa dữ liệu data về đoạn [0,1]
scaler = MinMaxScaler(feature_range=(0, 1))
X = scaler.fit_transform(X)

print("Kích thước X:", X.shape, "- Kích thước y:", y.shape)
print("Giá trị nhỏ nhất:", X.min(), "- Giá trị lớn nhất:", X.max())

Họ và tên: Lê Quốc Trung - MSSV: 6451071082
Kích thước X: (150, 4) - Kích thước y: (150,)
Giá trị nhỏ nhất: 0.0 - Giá trị lớn nhất: 1.0


## c) Định nghĩa hàm `create_model`

- Model là Neural Network tương tự câu 1.h: 2 hidden layers và 1 output layer.
  - Hidden layer 1: 10 neuron, kích hoạt ReLU, `input_shape` là (4,) (4 thuộc tính).
  - Hidden layer 2: 20 neuron, kích hoạt ReLU.
  - Output layer: 3 neuron (số class), kích hoạt Softmax.
- Biên dịch mô hình sử dụng optimizer: `adam`; loss: `sparse_categorical_crossentropy`; metric: `accuracy`.

In [2]:
print("Họ và tên: Lê Quốc Trung - MSSV: 6451071082")

def create_model():
    model = keras.Sequential([
        layers.Dense(10, activation='relu', input_shape=(4,)),  # hidden layer 1
        layers.Dense(20, activation='relu'),                    # hidden layer 2
        layers.Dense(3, activation='softmax')                   # output layer (3 class)
    ])
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

create_model().summary()


Họ và tên: Lê Quốc Trung - MSSV: 6451071082


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 10)             │            50 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 20)             │           220 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            63 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 333 (1.30 KB)

 Trainable params: 333 (1.30 KB)

 Non-trainable params: 0 (0.00 B)

## d) Huấn luyện mô hình bằng k-fold cross validation (k = 10)

Trong quá trình huấn luyện, in ra kết quả accuracy, precision, recall, f1-score của từng fold và lưu lại mô hình sau mỗi fold.

In [3]:
print("Họ và tên: Lê Quốc Trung - MSSV: 6451071082")

kfold = KFold(n_splits=10, shuffle=True, random_state=42)

models = []               # lưu các mô hình đã huấn luyện
accuracy_per_fold = []
precision_per_fold = []
recall_per_fold = []
f1_per_fold = []

fold_no = 1
for train_index, val_index in kfold.split(X):
    print(f"\n===== Fold {fold_no}/10 =====")

    model = create_model()

    X_train_fold, X_val_fold = X[train_index], X[val_index]
    y_train_fold, y_val_fold = y[train_index], y[val_index]

    # Huấn luyện mô hình trên tập train của fold
    model.fit(X_train_fold, y_train_fold, epochs=100, batch_size=8, verbose=0)

    # Lưu mô hình sau mỗi fold
    models.append(model)

    # Dự đoán trên tập validation và tính các độ đo
    y_pred = np.argmax(model.predict(X_val_fold, verbose=0), axis=1)
    acc = accuracy_score(y_val_fold, y_pred)
    prec = precision_score(y_val_fold, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_val_fold, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_val_fold, y_pred, average='macro', zero_division=0)

    accuracy_per_fold.append(acc)
    precision_per_fold.append(prec)
    recall_per_fold.append(rec)
    f1_per_fold.append(f1)

    print(f"Fold {fold_no}: accuracy = {acc:.4f}, precision = {prec:.4f}, recall = {rec:.4f}, f1-score = {f1:.4f}")
    fold_no += 1

print("\n===== Kết quả từng fold =====")
for i in range(len(models)):
    print(f"Fold {i+1}: accuracy = {accuracy_per_fold[i]:.4f}, precision = {precision_per_fold[i]:.4f}, "
          f"recall = {recall_per_fold[i]:.4f}, f1-score = {f1_per_fold[i]:.4f}")

Họ và tên: Lê Quốc Trung - MSSV: 6451071082

===== Fold 1/10 =====
Fold 1: accuracy = 0.9333, precision = 0.9167, recall = 0.9444, f1-score = 0.9221

===== Fold 2/10 =====


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Fold 2: accuracy = 1.0000, precision = 1.0000, recall = 1.0000, f1-score = 1.0000

===== Fold 3/10 =====


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Fold 3: accuracy = 1.0000, precision = 1.0000, recall = 1.0000, f1-score = 1.0000

===== Fold 4/10 =====


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


... (7 outputs left out)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Fold 9: accuracy = 1.0000, precision = 1.0000, recall = 1.0000, f1-score = 1.0000

===== Fold 10/10 =====


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Fold 10: accuracy = 0.8000, precision = 0.8492, recall = 0.8492, f1-score = 0.8462

===== Kết quả từng fold =====
Fold 1: accuracy = 0.9333, precision = 0.9167, recall = 0.9444, f1-score = 0.9221
Fold 2: accuracy = 1.0000, precision = 1.0000, recall = 1.0000, f1-score = 1.0000
Fold 3: accuracy = 1.0000, precision = 1.0000, recall = 1.0000, f1-score = 1.0000
Fold 4: accuracy = 0.9333, precision = 0.9524, recall = 0.9333, f1-score = 0.9373
Fold 5: accuracy = 1.0000, precision = 1.0000, recall = 1.0000, f1-score = 1.0000
Fold 6: accuracy = 0.8667, precision = 0.8667, recall = 0.8889, f1-score = 0.8500
Fold 7: accuracy = 0.8667, precision = 0.9167, recall = 0.8333, f1-score = 0.8413
Fold 8: accuracy = 1.0000, precision = 1.0000, recall = 1.0000, f1-score = 1.0000
Fold 9: accuracy = 1.0000, precision = 1.0000, recall = 1.0000, f1-score = 1.0000
Fold 10: accuracy = 0.8000, precision = 0.8492, recall = 0.8492, f1-score = 0.8462
